<a href="https://colab.research.google.com/github/gitdd12/Integral_discovery/blob/ccr-206f948d-nkcq34/colab/count_stageA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 센다 실험 — 단계 A, **A6** (길이를 독립으로 뽑는 새 환경 + 고친 보너스) 시드 여러 개 (코랩 GPU)
**바뀐 점(2026-10-21):** 이전에는 두 장면의 전체 길이를 항상 같게 했는데, 그러면 화자와 판정자가 길이를 공유 정보로 써서 "같은 개수라도 길이마다 다른 기호"를 쓰는 편법이 가능했습니다(진단 결과). 이제 두 장면의 길이를 **독립으로** 뽑아서 그 편법을 막았습니다.

| 설정 | 내용 |
|---|---|
| **A6** | 새 환경 + 고친 보너스(기호가 골고루 + 더미당 기호는 확실히, 첫 30%에 서서히 켬). 실패 중심 출제 없음 |
| **A7** (선택, 아래 EXPERIMENTS 에서 주석 해제) | A6 + 실패 중심 출제 |

참고 (옛 환경): A2 완벽한 코드 1/3, A5 0/8.

**처음 한 번만:** 런타임 → 런타임 유형 변경 → GPU(T4). 위에서부터 셀을 차례로 실행.
**끊겼을 때:** 같은 셀들을 처음부터 다시 실행하면 드라이브의 체크포인트부터 이어갑니다. 이미 끝난 시드는 건너뜁니다.

## 1. GPU 확인과 구글 드라이브 연결

In [28]:
import torch
print('GPU 사용 가능:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/countworld_runs1'     # 결과가 저장되는 폴더 (바꿔도 됨)
os.makedirs(BASE, exist_ok=True); print(BASE)

GPU 사용 가능: True Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/countworld_runs1


## 2. 저장소 내려받기

In [29]:
import os, subprocess
REPO = 'gitdd12/integral_discovery'
BRANCH = 'ccr-206f948d-nkcq34'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    url = f'https://{token}@github.com/{REPO}.git'
except Exception:
    token = None
    url = f'https://github.com/{REPO}.git'
if not os.path.exists('/content/integral_discovery'):
    subprocess.run(['git','clone','-b',BRANCH,url,'/content/integral_discovery'], check=True)
else:
    subprocess.run(['git','-C','/content/integral_discovery','pull'], check=True)
%cd /content/integral_discovery
!git log --oneline | head -3

/content/integral_discovery
61c5763 결과 3-A6: 설정이 FOCUS=2(A5n)임을 사용자 확인
4cbf340 결과 3-A6: 새 환경 시드 3개(일관성 1.00, 완벽한 코드 0/3)
41dbfa8 진단: 전체 길이 공유 정보 구멍 발견, 두 장면 길이를 독립으로 수정, 코랩 노트북 A6, train_sup 버그 수정


## 3. 설정 (필요하면 여기만 바꾸세요)

In [30]:
SEEDS = list(range(1, 9))    # 시드 1~8
STEPS = 12000
MAX_PARALLEL = 3
EXPERIMENTS = {
    'A5': dict(USAGE='1.0', MI='1.0', RAMP='0.3', ENT='0.03', FOCUS='2'),
}

## 4. 학습 실행 (끝날 때까지 이 셀이 돕니다. 1분마다 진행 상황을 출력)

In [31]:
import os, subprocess, time, json, glob
jobs = []
for name, cfg in EXPERIMENTS.items():
    out = f'{BASE}/{name}'; os.makedirs(out, exist_ok=True)
    for s in SEEDS:
        if os.path.exists(f'{out}/s{s}.json'):
            continue
        env = dict(os.environ, OUT=out, LISTENERS='1', REFRESH='0', THREADS='1', DEVICE='cuda' if torch.cuda.is_available() else 'cpu', **cfg)
        jobs.append((name, s, env))
print('돌려야 할 학습:', len(jobs), '개')
running = []
def last_line(name, s):
    p = f'{BASE}/{name}/s{s}.log'
    if not os.path.exists(p): return '(시작 전)'
    L = [l for l in open(p).read().splitlines() if l.strip()]
    return L[-1][:100] if L else ''
t0 = time.time(); tick = 0
while jobs or running:
    running = [(n, s, p) for (n, s, p) in running if p.poll() is None]
    while jobs and len(running) < MAX_PARALLEL:
        n, s, env = jobs.pop(0)
        p = subprocess.Popen(['python', '-u', 'scripts/run_count_A.py', str(s), str(STEPS)], env=env,
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        running.append((n, s, p))
    if tick % 6 == 0:
        print(f'--- {int((time.time()-t0)/60)}분 경과, 실행 중 {len(running)}개, 대기 {len(jobs)}개')
        for n, s, _ in running:
            print(f'  {n} 시드{s}:', last_line(n, s))
    tick += 1
    time.sleep(10)
print('모두 끝났습니다')

돌려야 할 학습: 0 개
모두 끝났습니다


## 5. 결과 요약

In [ ]:
import json, glob, numpy as np
def summarize():
    for name in EXPERIMENTS:
        print(f'===== {name}')
        accs, perfect = [], 0
        for s in SEEDS:
            p = f'{BASE}/{name}/s{s}.json'
            if not os.path.exists(p):
                print(f'  시드{s}: 아직'); continue
            r = json.load(open(p))['report']; e5 = r['e5']; adj = r['e4_adjacent']
            ok = sum(v >= 0.9 for v in adj.values())
            merged = {}
            for k, v in e5['modal'].items(): merged.setdefault(tuple(v), []).append(int(k))
            merged = [g for g in merged.values() if len(g) > 1]
            good = (r['e1_train'] >= 0.95 and ok == 9 and e5['consistency'] >= 0.95 and e5['injective'] >= 0.95)
            perfect += good; accs.append(r['e1_train'])
            print(f"  시드{s}: 정확도 {r['e1_train']:.3f} | 빼둔쌍 {r['e1_heldout_pairs']:.3f} | n대n+1 ≥90% {ok}/9 | "
                  f"일관성 {e5['consistency']:.2f} 단사성 {e5['injective']:.2f} | 합쳐진 개수 {merged} | {'완벽' if good else ''}")
        if accs:
            print(f'  >> 끝난 시드 {len(accs)}개 중 완벽한 코드 {perfect}개 ({perfect/len(accs):.0%}), 정확도 평균 {np.mean(accs):.3f} ± {np.std(accs):.3f}')
            print('  (완벽 = 정확도≥0.95, 모든 n대n+1 ≥90%, 일관성·단사성 ≥0.95)')
summarize()

## 6. (선택) 결과를 저장소에 올리기
왼쪽 열쇠 아이콘(보안 비밀)에 `GITHUB_TOKEN`이 있고 쓰기 권한이 있을 때만 됩니다. 안 되면 5번 셀의 출력을 복사해서 대화에 붙여주셔도 됩니다.

In [ ]:
import shutil, subprocess
assert token, 'GITHUB_TOKEN 이 없습니다 (Secrets 에 추가하세요)'
dst = '/content/integral_discovery/runs3/colab'
for name in EXPERIMENTS:
    for f in glob.glob(f'{BASE}/{name}/s*.json') + glob.glob(f'{BASE}/{name}/s*.log'):
        os.makedirs(f'{dst}/{name}', exist_ok=True); shutil.copy(f, f'{dst}/{name}/')
cmds = [['git','config','user.email','colab@example.com'], ['git','config','user.name','colab'],
        ['git','add','runs3/colab'], ['git','commit','-m','코랩 단계 A5 시드 여러 개 결과'],
        ['git','push','origin',BRANCH]]
for c in cmds:
    print(subprocess.run(c, cwd='/content/integral_discovery', capture_output=True, text=True).stdout[-300:])

In [32]:
import json, os, time
for s in range(1, 9):
    p = f'{BASE}/A5/s{s}.json'
    if os.path.exists(p):
        r = json.load(open(p))['report']
        print(f"시드{s}: 파일 시각 {time.strftime('%m-%d %H:%M', time.localtime(os.path.getmtime(p)))} | 일관성 {r['e5']['consistency']:.2f}")
    else:
        print(f'시드{s}: 없음')

시드1: 파일 시각 10-20 11:59 | 일관성 1.00
시드2: 파일 시각 10-20 11:58 | 일관성 1.00
시드3: 파일 시각 10-20 11:58 | 일관성 1.00
시드4: 파일 시각 10-20 13:00 | 일관성 1.00
시드5: 파일 시각 10-20 13:00 | 일관성 0.87
시드6: 파일 시각 10-20 13:00 | 일관성 0.99
시드7: 파일 시각 10-20 13:18 | 일관성 1.00
시드8: 파일 시각 10-20 13:18 | 일관성 1.00


In [33]:
import glob, json, os, time
for p in sorted(glob.glob('/content/drive/MyDrive/**/s?.json', recursive=True)):
    try:
        r = json.load(open(p))['report']
        print(p, '|', time.strftime('%m-%d %H:%M', time.localtime(os.path.getmtime(p))), '| 일관성', round(r['e5']['consistency'], 2))
    except Exception:
        pass

/content/drive/MyDrive/countworld_runs/A5/s1.json | 10-20 10:27 | 일관성 0.78
/content/drive/MyDrive/countworld_runs/A5/s2.json | 10-20 10:27 | 일관성 0.57
/content/drive/MyDrive/countworld_runs/A5/s3.json | 10-20 10:25 | 일관성 0.61
/content/drive/MyDrive/countworld_runs/A5/s4.json | 10-20 10:51 | 일관성 0.79
/content/drive/MyDrive/countworld_runs/A5/s5.json | 10-20 10:54 | 일관성 0.67
/content/drive/MyDrive/countworld_runs/A5/s6.json | 10-20 10:54 | 일관성 0.85
/content/drive/MyDrive/countworld_runs/A5/s7.json | 10-20 11:09 | 일관성 0.78
/content/drive/MyDrive/countworld_runs/A5/s8.json | 10-20 11:11 | 일관성 0.54
/content/drive/MyDrive/countworld_runs1/A5/s1.json | 10-20 11:59 | 일관성 1.0
/content/drive/MyDrive/countworld_runs1/A5/s2.json | 10-20 11:58 | 일관성 1.0
/content/drive/MyDrive/countworld_runs1/A5/s3.json | 10-20 11:58 | 일관성 1.0
/content/drive/MyDrive/countworld_runs1/A5/s4.json | 10-20 13:00 | 일관성 1.0
/content/drive/MyDrive/countworld_runs1/A5/s5.json | 10-20 13:00 | 일관성 0.87
/content/drive/MyDrive/c

## 7. (선택) 진단: 같은 개수에서 기호가 장면의 길이에 따라 바뀌는가
결과가 이상하면 이 출력을 대화에 붙여주세요.


In [ ]:
for name in EXPERIMENTS:
    print('=====', name)
    !python scripts/diagnose_count.py {BASE}/{name} 1 2 3 2>&1 | grep -v Warn | head -50
